# Revalidation Reassignment — Delaware peek

Scope: individual -> group reassignment link (the Q4 "who bills under whom" file).
Source: `data/sources/provider_registry/reassignment/Revalidation_Reassignment_List_Sep_2026.zip` (~626 MB).
Reading: chunked pandas, filtered to `Individual State Code` = DE.

In [1]:
import zipfile
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ZIP = ROOT / "data/sources/provider_registry/reassignment/Revalidation_Reassignment_List_Sep_2026.zip"
MEMBER = "Revalidation_Reassignment_List_Sep_2026.csv"
STATE_COL, STATE = "Individual State Code", "DE"
ENC = "latin-1"  # CMS enrollment files are not reliably UTF-8
with zipfile.ZipFile(ZIP) as z, z.open(MEMBER) as f:
    sample = pd.read_csv(f, nrows=5, dtype=str, encoding=ENC)
print(sample.shape[1], "columns")
sample.head()

16 columns


,Group PAC ID,Group Enrollment ID,Group Legal Business Name,Group State Code,Group Due Date,Group Reassignments and Physician Assistants,Record Type,Individual PAC ID,Individual Enrollment ID,Individual NPI,Individual First Name,Individual Last Name,Individual State Code,Individual Specialty Description,Individual Due Date,Individual Total Employer Associations
0,3476607466,I20090810000434,NaN,NY,TBD,1,Reassignment,4880027259,I20191126001426,1043724263,Syeda,Abbas,NY,Optometry,TBD,1
1,9032145594,I20050715000357,NaN,VA,TBD,5,Reassignment,9335414499,I20181029001413,1871899922,Amir,Abbasi,VA,Podiatry,TBD,1
2,1658406871,I20100312000754,NaN,HI,TBD,1,Reassignment,8224388582,I20180906002598,1598242984,Kellie,Abe,HI,Optometry,TBD,2
3,2466731609,I20161111000247,NaN,MI,TBD,14,Reassignment,2769589761,I20070529000176,1568592046,Theresa,Abed,MI,Clinical Social Worker,TBD,1
4,6709062623,I20110615000792,NaN,NY,TBD,1,Reassignment,6305931072,I20070926001198,1437144359,Michelle,Abis,NY,Internal Medicine,TBD,1


In [2]:
# Scope to Delaware. Chunked scan (~626 MB).
parts = []
with zipfile.ZipFile(ZIP) as z, z.open(MEMBER) as f:
    for chunk in pd.read_csv(f, dtype=str, chunksize=200_000, encoding=ENC):
        parts.append(chunk[chunk[STATE_COL] == STATE])
de = pd.concat(parts, ignore_index=True)
de.shape

(13771, 16)

In [3]:
# Grouping signal: individuals -> groups.
print("rows:", len(de), "| unique Individual NPIs:", de["Individual NPI"].nunique(), "| unique Group PAC IDs:", de["Group PAC ID"].nunique())
print("links per individual:", round(len(de) / de["Individual NPI"].nunique(), 2))
de["Group Legal Business Name"].value_counts().head(15)

rows: 13771 | unique Individual NPIs: 8866 | unique Group PAC IDs: 1262
links per individual: 1.55


Group Legal Business Name
Christiana Care Health Services Inc         2767
Bayhealth Medical Center, Inc               1172
The Nemours Foundation                       526
Beebe Physician Network Inc.                 375
Anesthesia Services Pa                       345
Grow Healthcare Group Pa                     232
Northstar Anesthesia Of Delaware Llc         176
Tidalhealth Physician Network, Inc.          176
Apogee Medical Group Delaware Inc            166
Christiana Care Emergency Physicians Llc     157
Ati Holdings, Llc                            153
Bayhealth Emergency Physicians Llc           140
Christiana Care Quality Partners             114
Delaware Center For Digestive Care Llc       103
Dynamic Therapy Services Llc                 102
Name: count, dtype: int64

## Notes
- One row per individual-to-group reassignment. Many individual NPIs -> one `Group PAC ID` = they bill together (Q4).
- Carries both `Individual State Code` and `Group State Code`; a DE individual may bill under an out-of-state group.
- `Group PAC ID` = billing entity, not a physical site (HISTORY 002 caution).